"""
# 🏥 CT Pathology Detection - Quick Start Guide

Этот ноутбук демонстрирует базовое использование системы для выявления патологий на КТ снимках.

**Что вы узнаете:**
- Инициализация pipeline
- Обработка ZIP архивов с DICOM/NIfTI
- Анализ результатов
- Визуализация статистики

**Требования:**
- Запущен install.sh
- Модели скачаны в папку models/
- Есть тестовые данные в формате ZIP
"""

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

project_root = Path.cwd()

from ct_pathology.pipeline.core_pipeline import CTPathologyPipeline
from ct_pathology.pipeline.data_models import PipelineConfig

print("✅ Импорты загружены")


In [ ]:
# =============================================================================
# = КОНФИГУРАЦИЯ СИСТЕМЫ  =
# =============================================================================
# Конфигурация pipeline
config = PipelineConfig(
    ct_clip_checkpoint="models/CT_LiPro_v2.pt",
    catboost_model="models/catboost_pathology_classifier.cbm",
    text_prompt="chest computed tomography scan for pathology detection",
    device="cuda",  # Измените на "cpu" если нет GPU
    max_workers=2,  # Количество параллельных потоков
    log_level="INFO"
)

print("✅ Конфигурация создана")
print(f"   Модель CT-CLIP: {config.ct_clip_checkpoint}")
print(f"   Модель CatBoost: {config.catboost_model}")
print(f"   Устройство: {config.device}")

In [ ]:
# =============================================================================
# == ИНИЦИАЛИЗАЦИЯ PIPELINE == 
# =============================================================================
print("⏳ Инициализация pipeline (может занять 1-2 минуты)...")

pipeline = CTPathologyPipeline(config)

print("✅ Pipeline успешно инициализирован!")
print("   CT-CLIP модель загружена")
print("   CatBoost классификатор загружен")

In [ ]:
# ВАЖНО: Укажите пути к вашим ZIP архивам с DICOM или NIfTI файлами
# Пример структуры ZIP:
#   study.zip
#   ├── series1/
#   │   ├── 001.dcm
#   │   ├── 002.dcm
#   │   └── ...
#   └── series2/
#       ├── 001.dcm
#       └── ...

# Укажите пути к вашим данным
zip_paths = [
    "tests/test_data/test_nested_structure.zip",  # Замените на реальные пути
    "tests/test_data/test_nifti.zip",
    "tests/test_data/test_varying_sizes.zip"
]

existing_zips = [p for p in zip_paths if Path(p).exists()]

if len(existing_zips) == 0:
    print("⚠️  ВНИМАНИЕ: Не найдены ZIP файлы!")
    print("   Укажите корректные пути в переменной zip_paths")
else:
    print(f"✅ Найдено {len(existing_zips)} ZIP архивов для обработки:")
    for zip_path in existing_zips:
        size_mb = Path(zip_path).stat().st_size / (1024**2)
        print(f"   - {zip_path} ({size_mb:.1f} MB)")

In [ ]:
# =============================================================================
#  ОБРАБОТКА ДАННЫХ
# =============================================================================
if len(existing_zips) > 0:
    print("\n⏳ Начинаем обработку...")
    print(f"   Архивов для обработки: {len(existing_zips)}")
    
    # Обработка через pipeline
    results = pipeline.process_zip_archives(
        zip_paths=existing_zips,
        output_excel="quick_start_results.xlsx"
    )
    
    print("\n✅ ОБРАБОТКА ЗАВЕРШЕНА!")
    print(f"   📊 Обработано исследований: {len(results)}")
    print(f"   📁 Результаты сохранены: quick_start_results.xlsx")
else:
    print("❌ Нет данных для обработки. Добавьте ZIP файлы.")

In [ ]:
# =============================================================================
# АНАЛИЗ РЕЗУЛЬТАТОВ
# =============================================================================
if len(existing_zips) > 0:
    # Загрузка результатов
    df = pd.read_excel("quick_start_results.xlsx", sheet_name="Results")
    
    # Базовая статистика
    success_count = (df['processing_status'] == 'Success').sum()
    failed_count = (df['processing_status'] == 'Failure').sum()
    pathology_count = (df['pathology'] == 1).sum()
    normal_count = (df['pathology'] == 0).sum()
    
    print("\n" + "="*60)
    print("📊 СТАТИСТИКА ОБРАБОТКИ")
    print("="*60)
    print(f"✅ Успешно обработано: {success_count}/{len(df)}")
    print(f"❌ Ошибок обработки: {failed_count}/{len(df)}")
    print(f"\n🔬 РЕЗУЛЬТАТЫ КЛАССИФИКАЦИИ:")
    print(f"🔴 Патологий обнаружено: {pathology_count} ({pathology_count/len(df)*100:.1f}%)")
    print(f"🟢 Норма: {normal_count} ({normal_count/len(df)*100:.1f}%)")
    print(f"\n📈 Средняя вероятность патологии: {df['probability_of_pathology'].mean():.4f}")
    print(f"📉 Медиана: {df['probability_of_pathology'].median():.4f}")
    print("="*60)
    
    # Показываем первые результаты
    print("\n📋 Первые 5 результатов:")
    display(df[['series_uid', 'probability_of_pathology', 'pathology', 'processing_status']].head())

In [ ]:
# =============================================================================
# ВИЗУАЛИЗАЦИЯ
# =============================================================================
if len(existing_zips) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # График 1: Распределение вероятностей
    axes[0].hist(df['probability_of_pathology'], bins=20, edgecolor='black', alpha=0.7)
    axes[0].axvline(0.5, color='red', linestyle='--', label='Порог классификации')
    axes[0].set_xlabel('Вероятность патологии', fontsize=12)
    axes[0].set_ylabel('Количество исследований', fontsize=12)
    axes[0].set_title('Распределение вероятностей патологий', fontsize=14, fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # График 2: Pie chart результатов
    labels = ['Патология', 'Норма']
    sizes = [pathology_count, normal_count]
    colors = ['#ff6b6b', '#51cf66']
    explode = (0.1, 0)
    
    axes[1].pie(sizes, explode=explode, labels=labels, colors=colors,
                autopct='%1.1f%%', shadow=True, startangle=90)
    axes[1].set_title('Соотношение патологий и нормы', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('quick_start_visualization.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("✅ Визуализация сохранена: quick_start_visualization.png")

In [ ]:
# =============================================================================
# == ДЕТАЛЬНЫЙ АНАЛИЗ ПАТОЛОГИЙ
# =============================================================================
if len(existing_zips) > 0 and pathology_count > 0:
    print("\n" + "="*60)
    print("🔍 ДЕТАЛЬНЫЙ АНАЛИЗ ПАТОЛОГИЙ")
    print("="*60)
    
    # Фильтруем только патологии
    pathologies = df[df['pathology'] == 1].sort_values('probability_of_pathology', ascending=False)
    
    print(f"\n📌 Топ-5 исследований с наивысшей вероятностью патологии:")
    for idx, row in pathologies.head(5).iterrows():
        series_short = row['series_uid'][:40] + "..." if len(row['series_uid']) > 40 else row['series_uid']
        print(f"\n{idx+1}. Series: {series_short}")
        print(f"   Вероятность: {row['probability_of_pathology']:.4f} ({row['probability_of_pathology']*100:.2f}%)")
        print(f"   Статус: {row['processing_status']}")

In [ ]:
print("\n" + "="*60)
print("QUICK START ЗАВЕРШЁН!")
print("="*60)
print("\n Что было сделано:")
print("   1. Инициализирован pipeline с CT-CLIP и CatBoost")
print("   2. Обработаны ZIP архивы с медицинскими изображениями")
print("   3. Сгенерированы детальные результаты в Excel")
print("   4. Проведён анализ и визуализация данных")

print("\n📚 Следующие шаги:")
print("   - Изучите детальные результаты в quick_start_results.xlsx")
print("   - Проверьте лист 'Errors' для анализа ошибок")
print("   - Используйте лист 'Summary' для общей статистики")
print("   - Примените pipeline к вашим собственным данным")

print("\n📖 Дополнительная документация:")
print("   - notebooks/final.ipynb - полный эксперимент с обучением")
print("   - README.md - детальное описание системы")
print("   - src/pipeline/core_pipeline.py - API документация")

print("\n💡 Совет: Для обработки больших объёмов данных используйте:")
print("   - GPU для ускорения (device='cuda')")
print("   - Увеличьте max_workers для параллелизма")
print("   - Используйте batch обработку через glob")